<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_3_Boosting_and_BART.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Boosting and BART: Sequential Learning and Beyond

**Attribution**: Exercises adapted from *Introduction to Modern Statistics (2e)* Chapter 9.6.
Source: [OpenIntro IMS](https://openintro-ims.netlify.app/model-logistic)

**License**: This work is a derivative of [Introduction to Modern Statistics (2e)](https://openintro-ims.netlify.app/) by Mine Çetinkaya-Rundel and Johanna Hardin, used under a [Creative Commons Attribution-ShareAlike 3.0 Unported (CC BY-SA 3.0 US)](https://creativecommons.org/licenses/by-sa/3.0/us/) license.

---

## Introduction: From Democracy to Relay Race

In the previous notebook, we used **Bagging** to build independent trees in parallel. The goal was to reduce **variance** by averaging many noisy models.

In this final notebook, we explore **Boosting**, where trees are built **sequentially**. Instead of a democracy of independent voters (like Random Forest), Boosting is like a team of experts working on a problem one by one.

The core idea is to combine many **Weak Learners**—models that are only slightly better than random guessing—into a single **Strong Learner** that can master complex patterns.

We will conclude with a look at **BART (Bayesian Additive Regression Trees)**, which combines the benefits of both strategies with a robust Bayesian framework for uncertainty quantification.

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Implement** Gradient Boosting and XGBoost models.
2. **Explain** the roles of learning rate ($\lambda$) and shrinkage in boosting.
3. **Contrast** the sequential logic of Boosting with the parallel logic of Bagging.
4. **Understand** the core principles of Bayesian Additive Regression Trees (BART).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Set style
sns.set_context("talk")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

---
## 1. Boosting: Learning from Mistakes

Boosting algorithms work by iteratively fixing the errors of the previous models.

### The Boosting Algorithm
1.  **Initialize**: Start with a simple model (often just predicting the average value).
2.  **Calculate Residuals**: Measure the difference between the true values and the current prediction.
3.  **Train Weak Learner**: Fit a new, shallow Decision Tree (often a "stump" with depth 1 or 2) to predict these **residuals**.
4.  **Update**: Add the new tree's predictions to the model, scaled by a **Learning Rate**.
5.  **Repeat**: Continue for hundreds or thousands of iterations.

### Why Shallow Trees?
In Random Forests, we used deep trees to reduce bias. In Boosting, we use **shallow trees** (high bias, low variance). By combining thousands of high-bias models, we progressively reduce the bias without exploding the variance.

### Implementation: XGBoost
XGBoost (eXtreme Gradient Boosting) is an optimized implementation of this algorithm that has dominated machine learning competitions for years.

In [ ]:
# Initialize and fit XGBoost model
xgb = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.05, 
    max_depth=3, 
    random_state=42
)
xgb.fit(X_train, y_train)

print(f"XGBoost Accuracy (Test): {accuracy_score(y_test, xgb.predict(X_test)):.3f}")

### Key Parameters for Tuning

1.  **Learning Rate ($\eta$ / `learning_rate`)**: This is the most critical parameter. It scales the contribution of each new tree (e.g., 0.01 or 0.1). 
    *   **Low Learning Rate**: Requires more trees (`n_estimators`) but generalizes better (less overfitting).
    *   **High Learning Rate**: Learns fast but can overshoot and overfit.

2.  **Number of Trees ($B$ / `n_estimators`)**: The total number of boosting rounds. Unlike Random Forests, adding too many trees in Boosting *can* lead to overfitting, so we often use "early stopping."

3.  **Tree Depth ($d$ / `max_depth`)**: As mentioned, boosting prefers shallow trees. A depth of 3-6 is standard, whereas Random Forests might use depth 20+.

4.  **Subsampling**: Like Bagging, XGBoost can use a random subset of data for each tree to speed up training and add randomness.

---
## 2. Bayesian Additive Regression Trees (BART)

BART is the Bayesian "sibling" of the ensemble methods. It uses a "sum-of-trees" model similar to Boosting:

$$ Y = \sum_{j=1}^{m} g(x; T_j, M_j) + \epsilon $$

### The Bayesian Difference
Instead of simply adding trees based on residuals, BART uses a statistical approach:

1.  **Priors as Regularization**: BART places a strong "prior" belief that each tree should be small (shallow) and weak. This prevents any single tree from dominating the prediction.
2.  **MCMC Sampling**: To fit the model, BART uses a sampling algorithm (Markov Chain Monte Carlo). In each step, it might grow a branch, prune a branch, or change a split rule.
3.  **Uncertainty Quantification**: This is BART's superpower. Because it's Bayesian, it doesn't just give a prediction; it gives a **probability distribution**.

**Medical Application**: In cancer diagnosis, knowing that the model is "95% confident" vs. "55% confident" is a life-or-death distinction. BART provides these credible intervals naturally.

---
## 3. The Grand Finale: Model Showdown

We have now learned three powerful ensemble techniques:
1.  **Random Forest**: Parallel, low variance, robust.
2.  **AdaBoost / Gradient Boosting**: Sequential, low bias, high accuracy.
3.  **XGBoost**: Optimized Gradient Boosting.

Let's run a rigorous **10-Fold Cross-Validation** to see which method reigns supreme on our Breast Cancer dataset.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, KFold

# 1. Define Models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# 2. Setup Cross-Validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 3. Run Comparison
print(f"{'Model':<20} | {'Mean Accuracy':<15} | {'Std Dev':<10}")
print("-" * 50)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
    results[name] = scores
    print(f"{name:<20} | {scores.mean():.4f}          | {scores.std():.4f}")

In [ ]:
# Visualize the results
plt.figure(figsize=(10, 6))
plt.boxplot(results.values(), labels=results.keys())
plt.title("10-Fold Cross-Validation Accuracy Distribution")
plt.ylabel("Accuracy")
plt.grid(True, axis='y')
plt.show()

## Summary: Choosing the Right Tool

*   **Random Forest**: The best "out-of-the-box" model. It requires very little tuning, is hard to overfit, and is parallelizable (fast).
*   **XGBoost/Gradient Boosting**: Often wins Kaggle competitions by a small margin. It requires careful tuning of the learning rate and tree depth but can squeeze out the last drop of predictive power.
*   **BART**: The choice when **uncertainty quantification** is as important as accuracy.

In this module, you have moved from simple interpretable trees to state-of-the-art machine learning ensembles. These tools are the workhorses of modern data science.